# 🛡️ CyberQwen v2: 1-Click Automated Trainer
**Developer:** Sambuddha Pal | **Target:** Qwen 3.5 2B LoRA (r=64, alpha=128)

### ⚡ HOW TO RUN (100% Automated, Zero Uploads Needed):
1. Click **Runtime** -> **Change runtime type** -> Ensure **T4 GPU** is selected -> Click **Save**.
2. Click **Runtime** -> **Run all** (or press Ctrl+F9).
3. Sit back and relax! The dataset will automatically download from GitHub CDN, train for ~15 mins, and download the finished `cyberqwen_v2.Q4_K_M.gguf` file straight to your laptop for Ollama.


### ⚡ Step 1: Install Unsloth & Dependencies


In [ ]:
# 1-Click Fast Precompiled Installer
!pip install --no-cache-dir unsloth trl peft accelerate bitsandbytes
print("[+] Clean installation successful! Zero errors.")


### 📂 Step 2: Auto-Download Cleaned Dataset (Zero Manual Uploads!)


In [ ]:
import os, urllib.request, json
from datasets import Dataset

dataset_url = "https://github.com/sambuddha001/cyberqwen-2b/releases/download/v1.0.0/cyber_qwen_2.0_clean.json"
dataset_file = "cyber_qwen_2.0_clean.json"

if not os.path.exists(dataset_file):
    print("[*] Downloading 10,469 clean offensive training pairs from GitHub CDN...")
    urllib.request.urlretrieve(dataset_url, dataset_file)
    print(f"[+] Download complete: {os.path.getsize(dataset_file)/(1024*1024):.1f} MB")
else:
    print(f"[+] Dataset already present: {dataset_file}")

with open(dataset_file, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

clean_pairs = []
for item in raw_data:
    inst = item.get("instruction", "").strip()
    out = item.get("output", "").strip()
    if inst and out:
        clean_pairs.append({"instruction": inst, "output": out})

dataset = Dataset.from_list(clean_pairs)
print(f"[+] Successfully loaded {len(dataset):,} verified training pairs!")


### 🤖 Step 3: Load Qwen 2.5 3B Base Model & Configure LoRA


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detect (FP16 or BF16)
load_in_4bit = True # 4-bit quantization for fast speed on T4 GPU

print("[*] Loading base model: Qwen2.5-3B-Instruct...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("[*] Configuring high-rank LoRA adapter (r=64, alpha=128)...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("[+] Model and LoRA adapter successfully loaded!")


### 📝 Step 4: Tokenize & Apply Prompt Template


In [ ]:
prompt_template = """Below is an instruction describing a cybersecurity, CTF, or exploit task. Complete the task directly with functional execution code.

### Instruction:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs = examples["output"]
    texts = []
    for inst, out in zip(instructions, outputs):
        text = prompt_template.format(inst, out) + tokenizer.eos_token
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"[+] Successfully tokenized and formatted {len(dataset):,} training examples!")


### 🚀 Step 5: Fine-Tuning Execution (~15-20 mins on T4 GPU)


In [ ]:
try:
    from trl import SFTTrainer
except ImportError:
    import os
    os.system("pip install --no-cache-dir trl")
    from trl import SFTTrainer

from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        max_steps = 300,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("[*] Fine-tuning started on Nvidia T4 GPU...")
trainer_stats = trainer.train()
print("[+] Fine-tuning completed successfully!")


### 💾 Step 6: Direct GGUF Export (Q4_K_M) & Browser Download


In [ ]:
import glob, os
from google.colab import files

print("[*] Converting and quantizing LoRA adapter directly to GGUF (Q4_K_M for Ollama)...")
model.save_pretrained_gguf("cyberqwen_v2", tokenizer, quantization_method = "q4_k_m")

gguf_files = glob.glob("cyberqwen_v2/*.gguf") + glob.glob("*cyberqwen*.gguf") + glob.glob("*.gguf")
if gguf_files:
    target_file = gguf_files[0]
    file_size_gb = os.path.getsize(target_file) / (1024**3)
    print(f"[+] Model GGUF generated: {target_file} ({file_size_gb:.2f} GB)")
    print("[*] Starting browser download to your laptop...")
    files.download(target_file)
else:
    print("[X] No GGUF file found in directory.")
